# Jurisdictional Complexity & Fire Response Analysis for Tribal Lands

**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** Census TIGER AIANNH, BLM Surface Management Agency, MTBS

## Overview
Fast-moving fires do not respect jurisdictional boundaries, but emergency response
systems do. This notebook analyzes the governance landscape on and around Tribal lands
by mapping land ownership mosaics (BLM SMA dataset), quantifying jurisdictional
fragmentation, and correlating complexity with historical fire frequency derived
from MTBS data.

## Research Questions
- Which Tribal lands have the most fragmented land ownership patterns?
- Does ownership fragmentation correlate with fire frequency?
- What does the ownership mosaic look like within and around each Tribal land unit?

## Outputs
- Fragmentation index per Tribal land unit
- Ownership diversity charts and maps
- Correlation analysis: fragmentation vs. fire frequency
- Complexity vs. risk quadrant analysis

## Data Sovereignty Note
> Tribal boundary data comes from Census TIGER AIANNH — federal statistical
> boundaries that may not reflect Tribal Nations' own definitions of territory.
> Land ownership data comes from the BLM Surface Management Agency dataset,
> which reflects federal administrative designations, not Tribal sovereignty.
> All data is real; no synthetic data is used.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from collections import defaultdict
from datetime import datetime

import contextily as ctx
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.validation import make_valid

from src.data import constants, loaders, validators
from src.data.constants import PRIMARY_TRIBES
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import (
    generate_citations,
    print_data_acknowledgment,
)
from src.viz import charts, styles

styles.apply_mpl_style()
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Output dir: {constants.OUTPUTS_DIR}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Data sovereignty acknowledgment 
print_data_acknowledgment(source_keys=["census_aiannh", "blm_sma", "mtbs"])

## Load Tribal Land Boundaries

In [ ]:
# Census TIGER AIANNH 
all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh",
    required_columns=["geometry", "NAME"],
)

tribal_lands = all_tribal[
    all_tribal["NAME"].isin(PRIMARY_TRIBES)
].copy().reset_index(drop=True)

# Dissolve non-contiguous parcels, repair geometry, clip to CONUS
tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False).reset_index(drop=True)
tribal_lands["geometry"] = tribal_lands.geometry.apply(
    lambda g: make_valid(g) if g is not None else g
)
CONUS = geo_utils.bbox_geodataframe((-127, 24, -65, 50)).geometry.iloc[0]
tribal_lands = tribal_lands[
    tribal_lands.geometry.notnull() &
    tribal_lands.geometry.is_valid &
    tribal_lands.intersects(CONUS)
].copy().reset_index(drop=True)

print(f"Tribal land units loaded: {len(tribal_lands)}")
print(tribal_lands[["NAME"]].to_string(index=False))

# Study area bounding box — used for all spatial queries
BUFFER_DEG = 0.5
bounds = tribal_lands.total_bounds
STUDY_BBOX = (
    bounds[0] - BUFFER_DEG,
    bounds[1] - BUFFER_DEG,
    bounds[2] + BUFFER_DEG,
    bounds[3] + BUFFER_DEG,
)
print(f"\nStudy bbox: {tuple(round(x, 2) for x in STUDY_BBOX)}")

## Load Land Ownership Data

Land ownership comes from the BLM Surface Management Agency (SMA) dataset,
which maps federal administrative surface ownership across CONUS including
BLM, USFS, NPS, FWS, BIA, DOD, state, and private lands.

**Note:** SMA reflects federal administrative designations. It does not
represent Tribal sovereignty or the full extent of Tribal stewardship.

In [ ]:
# BLM SMA (process one Tribal land at a time to avoid large bbox failures) 
# The SMA dataset is dense; querying per-tribe bbox is more reliable.

all_sma = []
BUF = 0.3  # degrees buffer around each Tribal land for context

for _, tribe in tribal_lands.iterrows():
    tb   = tribe.geometry.bounds
    bbox = (tb[0]-BUF, tb[1]-BUF, tb[2]+BUF, tb[3]+BUF)
    try:
        sma = loaders.load_blm_sma(bbox=bbox)
        sma["tribal_name"] = tribe["NAME"]
        all_sma.append(sma)
        print(f"  {tribe['NAME']}: {len(sma):,} SMA features")
    except Exception as e:
        print(f"  {tribe['NAME']}: failed — {e}")

if all_sma:
    ownership = pd.concat(all_sma, ignore_index=True)
    ownership = gpd.GeoDataFrame(ownership, crs=constants.CRS_GEOGRAPHIC)
    print(f"\nTotal SMA features loaded: {len(ownership):,}")
    if "ADMIN_AGENCY_CODE" in ownership.columns:
        print("\nAgency breakdown:")
        print(ownership["ADMIN_AGENCY_CODE"].value_counts().head(10).to_string())
else:
    raise RuntimeError(
        "No BLM SMA features loaded. "
        "Check network access to gis.blm.gov."
    )

## Load Fire History (MTBS)
Fire frequency per Tribal land is derived from MTBS perimeters (1984–present)
and used as the fire risk proxy for the complexity correlation analysis.

In [ ]:
# MTBS fire perimeters 
MTBS_LOCAL = constants.RAW_DIR / "mtbs_perimeters" / "mtbs_perims_DD.shp"

if MTBS_LOCAL.exists():
    fire_gdf = gpd.read_file(MTBS_LOCAL)
    fire_gdf.columns = fire_gdf.columns.str.lower()
    fire_gdf["ig_date"]    = pd.to_datetime(fire_gdf["ig_date"], errors="coerce")
    fire_gdf["fire_year"]  = fire_gdf["ig_date"].dt.year
    fire_gdf["fire_size_acres"] = fire_gdf.get("burnbndac", pd.Series(dtype=float))
    fire_gdf = fire_gdf.to_crs(constants.CRS_GEOGRAPHIC)
    print(f"MTBS perimeters loaded: {len(fire_gdf):,}")
else:
    print(
        "MTBS shapefile not found at data/raw/mtbs_perimeters/mtbs_perims_DD.shp.\n"
        "Download from https://www.mtbs.gov/direct-download and place there.\n"
        "Fire frequency analysis will be skipped."
    )
    fire_gdf = None

# Compute fire frequency per Tribal land via spatial join
if fire_gdf is not None:
    joined = gpd.sjoin(
        fire_gdf[["geometry", "fire_size_acres"]].to_crs("EPSG:5070"),
        tribal_lands[["NAME", "geometry"]].to_crs("EPSG:5070"),
        how="inner",
        predicate="intersects",
    )
    fire_freq = (
        joined.groupby("NAME")
        .agg(
            fire_count=("fire_size_acres", "count"),
            total_acres_burned=("fire_size_acres", "sum"),
        )
        .reset_index()
    )
    # Normalize to 0–10 risk score
    fire_freq["fire_risk_score"] = (
        fire_freq["fire_count"] / fire_freq["fire_count"].max() * 10
    ).round(2)
    tribal_lands = tribal_lands.merge(fire_freq, on="NAME", how="left")
    tribal_lands["fire_count"]       = tribal_lands["fire_count"].fillna(0)
    tribal_lands["fire_risk_score"]  = tribal_lands["fire_risk_score"].fillna(0)
    print("\nFire frequency per Tribal land:")
    print(tribal_lands[["NAME", "fire_count", "fire_risk_score"]].to_string(index=False))
else:
    tribal_lands["fire_count"]      = np.nan
    tribal_lands["fire_risk_score"] = np.nan

## Jurisdictional Fragmentation Analysis

In [ ]:
def calculate_fragmentation_metrics(
    tribal_name: str,
    tribal_geom,
    sma_gdf: gpd.GeoDataFrame,
    agency_col: str = "ADMIN_AGENCY_CODE",
) -> dict:
    """
    Calculate land ownership fragmentation metrics for one Tribal land unit.

    Metrics
    -------
    num_agency_types  : distinct SMA agency codes within/adjacent to Tribal land
    num_features      : count of SMA polygons intersecting Tribal land
    shannon_diversity : -sum(p * log(p)) over agency proportions
    largest_owner_pct : % of features belonging to single largest agency
    fragmentation_idx : composite score 0–100 (higher = more fragmented)
    """
    local = sma_gdf[
        (sma_gdf["tribal_name"] == tribal_name) &
        sma_gdf.intersects(tribal_geom)
    ].copy()

    if local.empty or agency_col not in local.columns:
        return {
            "tribal_name":       tribal_name,
            "num_agency_types":  0,
            "num_features":      0,
            "shannon_diversity": 0.0,
            "largest_owner_pct": 0.0,
            "fragmentation_idx": 0.0,
        }

    counts = local[agency_col].value_counts()
    n      = len(local)
    props  = counts / n
    shannon = float(-(props * np.log(props.clip(lower=1e-9))).sum())
    max_shannon = np.log(len(counts)) if len(counts) > 1 else 1.0

    # Composite index — normalized components
    norm_types   = min(len(counts) / 8, 1.0) * 30
    norm_features = min(n / 200, 1.0) * 30
    norm_shannon = (shannon / max_shannon if max_shannon > 0 else 0) * 40
    frag_idx     = norm_types + norm_features + norm_shannon

    return {
        "tribal_name":       tribal_name,
        "num_agency_types":  int(len(counts)),
        "num_features":      int(n),
        "shannon_diversity": round(shannon, 3),
        "largest_owner_pct": round(float(counts.max() / n * 100), 1),
        "fragmentation_idx": round(frag_idx, 1),
        "agency_breakdown":  counts.to_dict(),
    }


frag_results = [
    calculate_fragmentation_metrics(row["NAME"], row.geometry, ownership)
    for _, row in tribal_lands.iterrows()
]
frag_df = pd.DataFrame(frag_results).drop(columns=["agency_breakdown"], errors="ignore")

# Merge with fire risk
complexity = tribal_lands[["NAME", "fire_count", "fire_risk_score"]].merge(
    frag_df, left_on="NAME", right_on="tribal_name", how="left"
).drop(columns=["tribal_name"], errors="ignore")

print("FRAGMENTATION ANALYSIS")
print("=" * 70)
print(
    complexity[[
        "NAME", "num_agency_types", "num_features",
        "shannon_diversity", "fragmentation_idx", "fire_risk_score",
    ]]
    .sort_values("fragmentation_idx", ascending=False)
    .to_string(index=False)
)

## Governance Complexity Score

In [ ]:
# Composite governance complexity score 
# Combines land ownership fragmentation (70%) with agency type diversity (30%).
# Fire authority boundaries and mutual aid agreement data are not available
# from a public API; those components require direct engagement with BIA
# Division of Fire Management and individual Tribal fire programs.

complexity["governance_complexity_score"] = (
    complexity["fragmentation_idx"] * 0.7 +
    (complexity["num_agency_types"].fillna(0) / 8 * 100) * 0.3
).round(1)

complexity["complexity_category"] = pd.cut(
    complexity["governance_complexity_score"],
    bins=[0, 35, 65, 100],
    labels=["Low", "Moderate", "High"],
)

print("GOVERNANCE COMPLEXITY RANKINGS")
print("=" * 70)
print(
    complexity[[
        "NAME", "fragmentation_idx", "num_agency_types",
        "governance_complexity_score", "complexity_category",
    ]]
    .sort_values("governance_complexity_score", ascending=False)
    .to_string(index=False)
)

## Complexity vs. Fire Risk Analysis

In [ ]:
# Correlation 
if complexity["fire_risk_score"].notna().sum() >= 3:
    corr = complexity[["fire_risk_score", "governance_complexity_score"]].corr().iloc[0, 1]
    print(f"Correlation (fire risk vs. governance complexity): {corr:.3f}")
    if abs(corr) < 0.3:
        print("Weak: complexity issues exist across all Tribal lands regardless of risk level.")
    elif abs(corr) < 0.7:
        print("Moderate: some relationship between risk and governance complexity.")
    else:
        print("Strong: high-risk areas tend toward more complex governance.")
else:
    corr = None
    print("Insufficient fire data for correlation. Ensure MTBS file is loaded.")

In [ ]:
# Quadrant analysis
if complexity["fire_risk_score"].notna().sum() >= 2:
    risk_med       = complexity["fire_risk_score"].median()
    complexity_med = complexity["governance_complexity_score"].median()

    def _quadrant(row):
        hi_risk = row["fire_risk_score"]            >= risk_med
        hi_comp = row["governance_complexity_score"] >= complexity_med
        if hi_risk and hi_comp:  return "Q1: High Risk, High Complexity (CRITICAL)"
        if hi_risk:              return "Q4: High Risk, Low Complexity (Well-positioned)"
        if hi_comp:              return "Q2: Low Risk, High Complexity (Inefficient)"
        return                          "Q3: Low Risk, Low Complexity (Manageable)"

    complexity["quadrant"] = complexity.apply(_quadrant, axis=1)

    print("QUADRANT DISTRIBUTION")
    print("=" * 60)
    print(complexity["quadrant"].value_counts().to_string())

    critical = complexity[complexity["quadrant"] == "Q1: High Risk, High Complexity (CRITICAL)"]
    if not critical.empty:
        print(f"\nCRITICAL areas ({len(critical)}):")
        print(critical[["NAME", "fire_risk_score", "governance_complexity_score"]].to_string(index=False))
else:
    complexity["quadrant"] = "N/A: fire data not loaded"
    risk_med = complexity_med = None

## Visualizations

In [ ]:
# Fragmentation dashboard 
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

def _hbar(ax, col, title, xlabel, color_fn=None):
    s = complexity.sort_values(col, ascending=True)
    colors = color_fn(s[col]) if color_fn else styles.FIRE_ORANGE
    ax.barh(s["NAME"], s[col], color=colors, alpha=0.82)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel)
    sns.despine(ax=ax)

_hbar(axes[0, 0], "num_agency_types", "Land Ownership Agency Types",
      "Distinct Agency Types",
      lambda v: [styles.FIRE_ORANGE] * len(v))

_hbar(axes[0, 1], "shannon_diversity", "Ownership Diversity (Shannon Index)",
      "Shannon Diversity",
      lambda v: [styles.SKY_BLUE] * len(v))

_hbar(axes[1, 0], "fragmentation_idx", "Land Fragmentation Index",
      "Fragmentation Index (0–100)",
      lambda v: [
          styles.SAGE_GREEN if x < 35 else
          styles.FIRE_ORANGE if x < 65 else
          styles.EMBER_RED
          for x in v
      ])

_hbar(axes[1, 1], "governance_complexity_score",
      "Governance Complexity Score", "Score (0–100)",
      lambda v: [
          styles.SAGE_GREEN if x < 35 else
          styles.FIRE_ORANGE if x < 65 else
          styles.EMBER_RED
          for x in v
      ])

plt.suptitle(
    "Jurisdictional Fragmentation Metrics\nBased on BLM Surface Management Agency Data",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/fragmentation_dashboard.png")
plt.show()

In [ ]:
# Complexity vs. fire risk scatter 
if complexity["fire_risk_score"].notna().sum() >= 2 and risk_med is not None:
    quadrant_colors = {
        "Q1: High Risk, High Complexity (CRITICAL)":      styles.EMBER_RED,
        "Q2: Low Risk, High Complexity (Inefficient)":    styles.FIRE_ORANGE,
        "Q3: Low Risk, Low Complexity (Manageable)":      styles.SAGE_GREEN,
        "Q4: High Risk, Low Complexity (Well-positioned)": styles.SKY_BLUE,
    }

    fig, ax = plt.subplots(figsize=(12, 8))

    for quad, color in quadrant_colors.items():
        sub = complexity[complexity["quadrant"] == quad]
        if sub.empty:
            continue
        ax.scatter(
            sub["governance_complexity_score"],
            sub["fire_risk_score"],
            c=color, s=200, alpha=0.8,
            edgecolors=styles.CHARCOAL, linewidth=1.2,
            label=quad,
        )
        for _, row in sub.iterrows():
            ax.annotate(
                row["NAME"].split()[0],
                (row["governance_complexity_score"], row["fire_risk_score"]),
                xytext=(5, 5), textcoords="offset points", fontsize=8,
            )

    ax.axvline(complexity_med, color="gray", linestyle="--", alpha=0.5)
    ax.axhline(risk_med,       color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Governance Complexity Score", fontsize=12)
    ax.set_ylabel("Fire Risk Score (MTBS frequency)", fontsize=12)
    ax.set_title(
        "Jurisdictional Complexity vs. Fire Risk\n"
        "Fast fires do not respect boundaries, but response systems do.",
        fontsize=13, fontweight="bold",
    )
    if corr is not None:
        ax.text(
            0.02, 0.97, f"r = {corr:.3f}",
            transform=ax.transAxes, fontsize=10,
            va="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
        )
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
    sns.despine(ax=ax)
    plt.tight_layout()
    charts.save_figure(fig, "outputs/figures/complexity_vs_risk.png")
    plt.show()
else:
    print("Scatter plot skipped: fire risk data not available.")

In [ ]:
# Per-Tribe ownership maps
# Shows the SMA ownership mosaic within and around each Tribal land.

AGENCY_COLORS = {
    "BLM":  "#DAA520",
    "USFS": "#228B22",
    "NPS":  "#8B4513",
    "FWS":  "#4682B4",
    "BIA":  styles.FIRE_ORANGE,
    "DOD":  "#708090",
    "STA":  "#DC143C",   # State
    "PVT":  "#C0C0C0",   # Private
}
DEFAULT_COLOR = "#EEEEEE"

n_tribes = len(tribal_lands)
ncols, nrows = 2, (n_tribes + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 5))
axes = axes.flatten()

for idx, (_, tribe_row) in enumerate(tribal_lands.iterrows()):
    ax   = axes[idx]
    name = tribe_row["NAME"]
    tb   = tribe_row.geometry.bounds

    local_sma = ownership[
        (ownership["tribal_name"] == name) &
        ownership.intersects(tribe_row.geometry)
    ].to_crs(3857)

    tribe_3857 = gpd.GeoDataFrame(
        geometry=[tribe_row.geometry], crs=constants.CRS_GEOGRAPHIC
    ).to_crs(3857)

    # Plot ownership polygons
    if not local_sma.empty and "ADMIN_AGENCY_CODE" in local_sma.columns:
        for agency, color in AGENCY_COLORS.items():
            sub = local_sma[local_sma["ADMIN_AGENCY_CODE"] == agency]
            if not sub.empty:
                sub.plot(ax=ax, color=color, alpha=0.55, edgecolor="none")
        # Anything not in our color map
        other = local_sma[~local_sma["ADMIN_AGENCY_CODE"].isin(AGENCY_COLORS)]
        if not other.empty:
            other.plot(ax=ax, color=DEFAULT_COLOR, alpha=0.4, edgecolor="none")

    # Tribal boundary overlay
    tribe_3857.boundary.plot(
        ax=ax, color=styles.CHARCOAL, linewidth=2, zorder=5
    )

    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.4)
    except Exception:
        pass

    frag_row = complexity[complexity["NAME"] == name]
    frag_val = frag_row["fragmentation_idx"].iloc[0] if not frag_row.empty else 0
    ax.set_title(f"{name}\nFragmentation: {frag_val:.0f}/100",
                 fontsize=9, fontweight="bold")
    ax.set_axis_off()

for ax in axes[n_tribes:]:
    ax.set_visible(False)

legend_handles = [
    mpatches.Patch(facecolor=c, alpha=0.6, label=a)
    for a, c in AGENCY_COLORS.items()
] + [mpatches.Patch(facecolor=DEFAULT_COLOR, alpha=0.5, label="Other")]

fig.legend(
    handles=legend_handles, loc="lower center",
    ncol=len(legend_handles), fontsize=8,
    bbox_to_anchor=(0.5, 0.01),
)
plt.suptitle(
    "Land Ownership Mosaic — BLM Surface Management Agency Data",
    fontsize=13, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
charts.save_figure(fig, "outputs/figures/ownership_mosaic_maps.png")
plt.show()

## Exports

In [ ]:
# Tabular exports 
exports = {
    "jurisdictional_complexity.csv": complexity.drop(
        columns=["geometry"], errors="ignore"
    ),
}
for fname, df in exports.items():
    path = constants.OUTPUTS_DIR / fname
    df.to_csv(path, index=False)
    print(f"Exported → {path.relative_to(REPO_ROOT)}")

# Spatial export 
tribal_with_complexity = tribal_lands.merge(
    complexity.drop(columns=["geometry"], errors="ignore"),
    on="NAME", how="left",
)
geo_path = constants.OUTPUTS_DIR / "tribal_complexity_analysis.geojson"
tribal_with_complexity[
    ["NAME", "fire_count", "fire_risk_score",
     "fragmentation_idx", "governance_complexity_score",
     "complexity_category", "quadrant", "geometry"]
].to_file(geo_path, driver="GeoJSON")
print(f"Exported → {geo_path.relative_to(REPO_ROOT)}")

## Summary and Findings

*(Fill in after running with real data.)*

!uestions to address in the narrative:
- Which Tribal lands show the highest fragmentation and why?
  (Checkerboard ownership from allotment-era land policies is a common driver.)
- Is there a meaningful correlation between ownership fragmentation and
  historical fire frequency? What does the direction of that correlation suggest?
- What are the limitations? BLM SMA reflects federal administrative designations,
  not the full picture of Tribal stewardship or fire authority jurisdiction.
  Mutual aid agreement data and fire authority boundaries require direct
  engagement with BIA Division of Fire Management and Tribal fire programs.
- This analysis provides a data-driven starting point for policy conversations
  about jurisdictional barriers, it does not prescribe governance solutions,
  which must be developed by and with Tribal Nations on their own terms.

---
## References

In [ ]:
print(generate_citations(["census_aiannh", "blm_sma", "mtbs"]))